In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter
from sksurv.metrics import brier_score, integrated_brier_score

warnings.filterwarnings('ignore')

# =====================================================================
# 1. PREPROCESSING & DATA LOADING
# =====================================================================
times = pd.read_csv("IntermediateResults/Patient_Atlas.csv")
times['Patient_ID'] = times['Patient_ID'].astype(str).str.strip()

patient_vals = pd.read_excel("InputData/venex_cohort_OS_labs_with_mutations.xlsx")
patient_vals['Study_subject'] = patient_vals['Study_subject'].astype(str).str.strip()

params = pd.read_csv("IntermediateResults/venex_weighted_params.csv")
params['ID'] = params['ID'].astype(str).str.strip()

# Merge static clinical baseline with static parameters
merged = pd.merge(patient_vals, params, left_on='Study_subject', right_on='ID', how='inner')

# Merge static clinical baseline with dynamic time-series atlas
merged_time = pd.merge(
    patient_vals,
    times,
    left_on='Study_subject',
    right_on='Patient_ID',
    how='inner'
)

# Numeric conversions for survival endpoints
merged['OS_time'] = pd.to_numeric(merged['OS_time'], errors='coerce')
merged['OS_status'] = pd.to_numeric(merged['OS_status'], errors='coerce')

merged_time['OS_time'] = pd.to_numeric(merged_time['OS_time'], errors='coerce')
merged_time['OS_status'] = pd.to_numeric(merged_time['OS_status'], errors='coerce')

# Filter complete cases across both sets
cols_static_raw = ['Study_subject', 'OS_time', 'OS_status', 'coef_weighted', 'pHSC_weighted']
cols_time_raw = [
    'Patient_ID', 'OS_time', 'OS_status', 'Predefined_Coef', 'Predefined_pHSC',
    'Fitted_DrugKill_LSC', 'Fitted_DrugKill_HSC'
]

clean_static = merged.dropna(subset=cols_static_raw).copy()
clean_time = merged_time.dropna(subset=cols_time_raw).copy()

common_ids = set(clean_static['Study_subject']).intersection(set(clean_time['Patient_ID']))
print(f"Cohort size across both modalities: {len(common_ids)} patients.")

df_static = clean_static[clean_static['Study_subject'].isin(common_ids)].sort_values('Study_subject').reset_index(drop=True)
df_time = clean_time[clean_time['Patient_ID'].isin(common_ids)].sort_values('Patient_ID').reset_index(drop=True)

# ---------------------------------------------------------------------
# STANDARDIZATION & FEATURE ENGINEERING
# ---------------------------------------------------------------------
# Static baseline standardization
df_static['coef_z'] = (df_static['coef_weighted'] - df_static['coef_weighted'].mean()) / (df_static['coef_weighted'].std() or 1.0)
df_static['pHSC_z'] = (df_static['pHSC_weighted'] - df_static['pHSC_weighted'].mean()) / (df_static['pHSC_weighted'].std() or 1.0)

# 1. Clip raw dynamic kill rates
df_time['Fitted_DrugKill_LSC_clean'] = df_time['Fitted_DrugKill_LSC'].clip(lower=1e-4, upper=5.0)
df_time['Fitted_DrugKill_HSC_clean'] = df_time['Fitted_DrugKill_HSC'].clip(lower=1e-4, upper=5.0)

# 2. Standardize dynamic ODE variables
df_time['Predefined_Coef_z'] = (df_time['Predefined_Coef'] - df_time['Predefined_Coef'].mean()) / (df_time['Predefined_Coef'].std() or 1.0)
df_time['Predefined_pHSC_z'] = (df_time['Predefined_pHSC'] - df_time['Predefined_pHSC'].mean()) / (df_time['Predefined_pHSC'].std() or 1.0)
df_time['Fitted_DrugKill_LSC_z'] = (df_time['Fitted_DrugKill_LSC_clean'] - df_time['Fitted_DrugKill_LSC_clean'].mean()) / (df_time['Fitted_DrugKill_LSC_clean'].std() or 1.0)
df_time['Fitted_DrugKill_HSC_z'] = (df_time['Fitted_DrugKill_HSC_clean'] - df_time['Fitted_DrugKill_HSC_clean'].mean()) / (df_time['Fitted_DrugKill_HSC_clean'].std() or 1.0)

static_cols = ['OS_time', 'OS_status', 'pHSC_z', 'coef_z']
time_cols = [
    'OS_time', 'OS_status', 'Predefined_pHSC_z', 'Predefined_Coef_z',
    'Fitted_DrugKill_LSC_z', 'Fitted_DrugKill_HSC_z'
]

df_static_model = df_static[static_cols].astype(float)
df_time_model = df_time[time_cols].astype(float)

df_static_model.index = df_static['Study_subject']
df_time_model.index = df_time['Patient_ID']

# =====================================================================
# 2. FIT COX PROPORTIONAL HAZARDS MODELS
# =====================================================================
formula_static = "pHSC_z + coef_z"
formula_time = "Predefined_Coef_z + Predefined_pHSC_z + Fitted_DrugKill_LSC_z * Fitted_DrugKill_HSC_z"

cph_static = CoxPHFitter(penalizer=0.01)
cph_static.fit(df_static_model, duration_col='OS_time', event_col='OS_status', formula=formula_static)

cph_time = CoxPHFitter(penalizer=0.01)
cph_time.fit(df_time_model, duration_col='OS_time', event_col='OS_status', formula=formula_time)

print(f"Static Model C-Index:    {cph_static.concordance_index_:.3f}")
print(f"Time-Series C-Index:     {cph_time.concordance_index_:.3f}")

## =====================================================================
# 3. IPCW BRIER SCORE OVER TIME (scikit-survival + safe integration)
# =====================================================================
def get_sksurv_array(df, time_col, event_col):
    return np.array(
        [(bool(e), t) for e, t in zip(df[event_col], df[time_col])],
        dtype=[('event', '?'), ('time', '<f8')]
    )

surv_train = get_sksurv_array(df_static_model, 'OS_time', 'OS_status')
surv_test = get_sksurv_array(df_time_model, 'OS_time', 'OS_status')

t_min = np.percentile(df_static_model['OS_time'], 10)
t_max = np.percentile(df_static_model['OS_time'], 85)
eval_times = np.linspace(t_min, t_max, 60)

pred_surv_static = cph_static.predict_survival_function(df_static_model, times=eval_times).T.values
pred_surv_time = cph_time.predict_survival_function(df_time_model, times=eval_times).T.values

# 1. Get the continuous curves using sksurv
times_static, bs_curve_static = brier_score(surv_train, surv_test, pred_surv_static, eval_times)
times_time, bs_curve_time = brier_score(surv_train, surv_test, pred_surv_time, eval_times)

# 2. the IBS (
integrate_func = getattr(np, 'trapezoid', getattr(np, 'trapz', None))

ibs_static = integrate_func(bs_curve_static, eval_times) / (eval_times[-1] - eval_times[0])
ibs_time = integrate_func(bs_curve_time, eval_times) / (eval_times[-1] - eval_times[0])

print(f"Static Model IBS:    {ibs_static:.3f}")
print(f"Time-Series IBS:     {ibs_time:.3f}")
# =====================================================================
# 4. DIGITAL TWIN CASE IDENTIFICATION
# =====================================================================
surv_static_730 = cph_static.predict_survival_function(df_static_model, times=[730]).values.ravel()
surv_time_730 = cph_time.predict_survival_function(df_time_model, times=[730]).values.ravel()

scanner_df = pd.DataFrame(index=df_time_model.index)
scanner_df['OS_time'] = df_time_model['OS_time']
scanner_df['OS_status'] = df_time_model['OS_status']
scanner_df['S_Static_2yr'] = surv_static_730
scanner_df['S_Time_2yr'] = surv_time_730
scanner_df['Shift_Optimism'] = scanner_df['S_Time_2yr'] - scanner_df['S_Static_2yr']

# Prioritize alive patients where Time-Series probability stays high AND well above Static
scanner_df['PanelA_Score'] = scanner_df['Shift_Optimism'] * (scanner_df['S_Time_2yr'] ** 2)

# Panel A: Alive patient with high sustained Time-Series survival & large upward shift
sub_A = scanner_df[(scanner_df['OS_time'] > 400) & (scanner_df['OS_status'] == 0)]
if sub_A.empty:
    sub_A = scanner_df[scanner_df['OS_status'] == 0]
id_A = (sub_A if not sub_A.empty else scanner_df).sort_values('PanelA_Score', ascending=False).index[0]

# Panel B: Deceased patient with strongest downward survival adjustment
sub_B = scanner_df[(scanner_df['OS_time'] >= 60) & (scanner_df['OS_time'] <= 400) & (scanner_df['OS_status'] == 1)]
id_B = (sub_B if not sub_B.empty else scanner_df).sort_values('Shift_Optimism', ascending=True).index[0]

# =====================================================================
# 5. GENERATE FIGURE
# =====================================================================
sns.set_theme(style="whitegrid", rc={"grid.color": "#f0f0f0", "grid.linestyle": "-"})
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['pdf.fonttype'] = 42

fig, axes = plt.subplots(1, 3, figsize=(24, 7.8))
plt.subplots_adjust(wspace=0.18)

c_static = "#0072B2"      # Okabe-Ito Deep Blue
c_dynamic = "#D55E00"     # Okabe-Ito Vermilion
c_neutral = "#222222"     # Dark charcoal

max_time = max(df_static_model['OS_time'].max(), df_time_model['OS_time'].max()) * 1.05

# ----------------- PANEL A: PATIENT ID_A (ALIVE) -----------------
ax_a = axes[0]
curve_stat_A = cph_static.predict_survival_function(df_static_model.loc[[id_A]])
curve_time_A = cph_time.predict_survival_function(df_time_model.loc[[id_A]])

ax_a.step(curve_stat_A.index.values, curve_stat_A.iloc[:, 0].values, where='post',
          color=c_static, linewidth=3.2, linestyle='--', label="Diagnostic ODE Model")
ax_a.step(curve_time_A.index.values, curve_time_A.iloc[:, 0].values, where='post',
          color=c_dynamic, linewidth=3.6, linestyle='-', label="Time-Series ODE Model")

os_day_A = df_time_model.loc[id_A, 'OS_time']
os_stat_A = df_time_model.loc[id_A, 'OS_status']
status_A = "Deceased" if os_stat_A == 1 else "Alive"
ax_a.axvline(x=os_day_A, color=c_neutral, linestyle='-.', linewidth=2.6,
             label=f"Last Follow-Up ({os_day_A:.0f} d - {status_A})")

eval_A = np.linspace(0, min(os_day_A, curve_stat_A.index.max()), 100)
s_sA = np.interp(eval_A, curve_stat_A.index.values, curve_stat_A.iloc[:, 0].values)
s_tA = np.interp(eval_A, curve_time_A.index.values, curve_time_A.iloc[:, 0].values)
ax_a.fill_between(eval_A, s_sA, s_tA, color=c_dynamic, alpha=0.18, label="Dynamic Upward Adjustment")

ax_a.text(-0.02, 1.05, "A", transform=ax_a.transAxes, fontsize=26, fontweight='bold', va='top', ha='right')

ax_a.set_xlabel("Time Since Treatment Initiation (Days)", fontsize=17, fontweight='bold', labelpad=10)
ax_a.set_ylabel("Predicted Overall Survival Probability", fontsize=17, fontweight='bold', labelpad=10)
ax_a.set_xlim(-10, max_time)
ax_a.set_ylim(-0.02, 1.02)
ax_a.tick_params(axis='both', labelsize=15)
ax_a.legend(loc='lower left', frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=13.5, borderpad=0.6)

# ----------------- PANEL B: PATIENT ID_B (DECEASED) -----------------
ax_b = axes[1]
curve_stat_B = cph_static.predict_survival_function(df_static_model.loc[[id_B]])
curve_time_B = cph_time.predict_survival_function(df_time_model.loc[[id_B]])

ax_b.step(curve_stat_B.index.values, curve_stat_B.iloc[:, 0].values, where='post',
          color=c_static, linewidth=3.2, linestyle='--', label="Diagnostic ODE Model")
ax_b.step(curve_time_B.index.values, curve_time_B.iloc[:, 0].values, where='post',
          color=c_dynamic, linewidth=3.6, linestyle='-', label="Time-Series ODE Model")

os_day_B = df_time_model.loc[id_B, 'OS_time']
os_stat_B = df_time_model.loc[id_B, 'OS_status']
status_B = "Deceased" if os_stat_B == 1 else "Alive"
ax_b.axvline(x=os_day_B, color=c_dynamic, linestyle=':', linewidth=2.8,
             label=f"Observed OS ({os_day_B:.0f} d - {status_B})")

eval_B = np.linspace(0, min(os_day_B, curve_stat_B.index.max()), 100)
s_sB = np.interp(eval_B, curve_stat_B.index.values, curve_stat_B.iloc[:, 0].values)
s_tB = np.interp(eval_B, curve_time_B.index.values, curve_time_B.iloc[:, 0].values)
ax_b.fill_between(eval_B, s_sB, s_tB, color=c_static, alpha=0.18, label="Dynamic Downward Adjustment")

ax_b.text(-0.02, 1.05, "B", transform=ax_b.transAxes, fontsize=26, fontweight='bold', va='top', ha='right')

ax_b.set_xlabel("Time Since Treatment Initiation (Days)", fontsize=17, fontweight='bold', labelpad=10)
ax_b.set_ylabel("Predicted Overall Survival Probability", fontsize=17, fontweight='bold', labelpad=10)
ax_b.set_xlim(-10, max_time)
ax_b.set_ylim(-0.02, 1.02)
ax_b.tick_params(axis='both', labelsize=15)
ax_b.legend(loc='lower left', frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=13.5, borderpad=0.6)

# ----------------- PANEL C: CONTINUOUS IPCW BRIER SCORE -----------------
ax_c = axes[2]
ax_c.plot(eval_times, bs_curve_static, color=c_static, linewidth=3.2, linestyle='--',
          label=f"Diagnostic ODE Model (IBS = {ibs_static:.3f})")
ax_c.plot(eval_times, bs_curve_time, color=c_dynamic, linewidth=3.6, linestyle='-',
          label=f"Time-Series ODE Model(IBS = {ibs_time:.3f})")

ax_c.text(-0.02, 1.05, "C", transform=ax_c.transAxes, fontsize=26, fontweight='bold', va='top', ha='right')

ax_c.set_xlabel("Time Since Treatment Initiation (Days)", fontsize=17, fontweight='bold', labelpad=10)
ax_c.set_ylabel("IPCW Brier Score (Prediction Error)", fontsize=17, fontweight='bold', labelpad=10)
ax_c.set_ylim(-0.005, max(max(bs_curve_static), max(bs_curve_time)) * 1.25)
ax_c.tick_params(axis='both', labelsize=15)
ax_c.legend(loc='upper left', frameon=True, facecolor='white', edgecolor='#cccccc', fontsize=13.5, borderpad=0.6)

for ax in axes:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

os.makedirs("Figures", exist_ok=True)
plt.savefig('Figures/Figure5.pdf', dpi=300, bbox_inches='tight', pad_inches=0.08)
plt.savefig('Figures/Figure5.png', dpi=300, bbox_inches='tight', pad_inches=0.08)
print(f"Figure successfully saved to Figures/Figure5.png and .pdf (Panel A: Patient {id_A}, Panel B: Patient {id_B})")
plt.show()